# Clean the Flexible Care Demand by Service LGA table from Excel

This notebook builds a simple data pipeline for the uploaded Excel file and only reads the required worksheet.
It cleans the flexible care table and reshapes the care type columns into a long-format table for loading into Postgres.


In [11]:
from pathlib import Path
import pandas as pd

INPUT_XLSX = Path("input/gen_data_flex_care_demand_by_service_loc.xlsx")
SHEET_NAME = "Table 5.1 (LGA)"

INPUT_XLSX, SHEET_NAME

(PosixPath('input/gen_data_flex_care_demand_by_service_loc.xlsx'),
 'Table 5.1 (LGA)')

In [12]:
raw_df = pd.read_excel(
    INPUT_XLSX,
    sheet_name=SHEET_NAME,
    header=None
)

raw_df.head(10)

,0,1,2,3
0,Table 5.1: Number of people using flexible car...,NaN,NaN,NaN
1,Return to contents,NaN,NaN,NaN
2,LGA code,LGA name,Transition Care Program (TPC),Short-Term Restorative Care (STRC)
3,10050,Albury,0,5
4,10750,Blacktown,0,25
5,11520,Canada Bay,94,0
6,11570,Canterbury-Bankstown,0,86
7,11650,Central Coast (NSW),128,49
8,12350,Cowra,0,1
9,12390,Dubbo,69,8


In [13]:
def clean_flex_care_demand(input_path: Path, sheet_name: str = "Table 5.1 (LGA)"):
    # Read only the required worksheet without assuming the header row position.
    df = pd.read_excel(input_path, sheet_name=sheet_name, header=None)

    # Find the real header row by matching the first column name.
    header_mask = (
        df.iloc[:, 0].astype("string").str.contains(r"^LGA code$", na=False)
    )

    if not header_mask.any():
        raise ValueError("Could not find the header row for the LGA table.")

    header_row_idx = header_mask[header_mask].index[0]

    # Rebuild the dataframe using the detected header row.
    df = df.iloc[header_row_idx + 1:].copy()
    df.columns = df.iloc[0:0].reindex(columns=df.columns).columns  # keep column count stable
    df.columns = pd.Index(pd.read_excel(input_path, sheet_name=sheet_name, header=None).iloc[header_row_idx].tolist())

    # Keep only the required columns from the worksheet.
    keep_columns = ["LGA code", "LGA name", "Transition Care Program (TPC)", "Short-Term Restorative Care (STRC)"]
    df = df[keep_columns].copy()

    # Drop completely empty rows.
    df = df.dropna(axis=0, how="all")

    # Rename columns.
    df = df.rename(columns={
        "LGA code": "lga_code",
        "LGA name": "lga_name",
        "Transition Care Program (TPC)": "tpc",
        "Short-Term Restorative Care (STRC)": "strc",
    })

    # Clean text columns: normalize whitespace and convert blanks to missing values.
    text_cols = ["lga_code", "lga_name", "tpc", "strc"]
    for col in text_cols:
        s = df[col].astype("string")
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
        s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df[col] = s

    # Remove repeated header rows if they appear in the body.
    df = df[df["lga_code"] != "LGA code"].copy()

    # Treat the code as strings.
    df["lga_code"] = (
        pd.to_numeric(df["lga_code"], errors="coerce")
        .astype("Int64").astype("string")
    )

    # Clean numeric columns.
    numeric_cols = ["tpc", "strc"]
    for col in numeric_cols:
        s = df[col].astype("string").str.replace(",", "", regex=False).str.strip()
        df[col] = pd.to_numeric(s, errors="coerce").astype("Int64")

    # Reshape to long format.
    df = df.melt(
        id_vars=["lga_code", "lga_name"],
        value_vars=["tpc", "strc"],
        var_name="care_type",
        value_name="people_count",
    )

    # Map care type values to readable labels.
    df["care_type"] = df["care_type"].replace({
        "tpc": "TPC",
        "strc": "STRC",
    })

    # Drop rows with missing key values.
    df = df.dropna(subset=["lga_code", "lga_name", "care_type", "people_count"]).copy()

    summary = {
        "rows": len(df),
        "unique_lgas": df["lga_code"].nunique(dropna=True),
        "care_types": sorted(df["care_type"].dropna().unique().tolist()),
        "null_people_count": int(df["people_count"].isna().sum()),
    }

    return df.reset_index(drop=True), summary

In [14]:
clean_df, summary = clean_flex_care_demand(INPUT_XLSX, SHEET_NAME)
summary

{'rows': 206,
 'unique_lgas': 103,
 'care_types': ['STRC', 'TPC'],
 'null_people_count': 0}

In [15]:
clean_df.head()

,lga_code,lga_name,care_type,people_count
0,10050,Albury,TPC,0
1,10750,Blacktown,TPC,0
2,11520,Canada Bay,TPC,94
3,11570,Canterbury-Bankstown,TPC,0
4,11650,Central Coast (NSW),TPC,128


In [16]:
print(f"Blank rows remaining: {int(clean_df.isna().all(axis=1).sum())}")
print(f"Exact duplicates remaining: {int(clean_df.duplicated().sum())}")
print(f"Null lga_code rows: {int(clean_df['lga_code'].isna().sum())}")
print(f"Null lga_name rows: {int(clean_df['lga_name'].isna().sum())}")
print(f"Data types:\n{clean_df.dtypes}")

Blank rows remaining: 0
Exact duplicates remaining: 0
Null lga_code rows: 0
Null lga_name rows: 0
Data types:
lga_code        string
lga_name        string
care_type          str
people_count     Int64
dtype: object


# Insert Clean Data into DB

In [17]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/flex_care_demand_by_service_lga.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


In [18]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
df_insert = clean_df.copy()

target_columns = [
    "lga_code",
    "lga_name",
    "care_type",
    "people_count",
]

df_insert = df_insert[target_columns].copy()

text_cols = [
    "lga_code",
    "lga_name",
    "care_type",
]

for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

int_cols = [
    "people_count",
]

for col in int_cols:
    df_insert[col] = pd.to_numeric(df_insert[col], errors="coerce").astype("Int64")

df_insert.head()

,lga_code,lga_name,care_type,people_count
0,10050,Albury,TPC,0
1,10750,Blacktown,TPC,0
2,11520,Canada Bay,TPC,94
3,11570,Canterbury-Bankstown,TPC,0
4,11650,Central Coast (NSW),TPC,128


In [19]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "flex_care_demand_by_service_lga",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert)} rows into flex_care_demand_by_service_lga")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM flex_care_demand_by_service_lga"))
    row_count = result.scalar()

print("Rows in table:", row_count)

Inserted 206 rows into flex_care_demand_by_service_lga
Rows in table: 206
